# 📊 Análisis Exploratorio de Datos Financieros (EDA)
### Portafolio de Data Analytics | Autor: Jose Longa (TotoBlunt)

El objetivo de este notebook es realizar un análisis cuantitativo y estadístico exhaustivo sobre el dataset de finanzas personales/familiares, respondiendo a preguntas clave de negocio:
1. **Concentración del Gasto (Pareto 80/20):** ¿Qué subconjunto de categorías explica la mayor parte del flujo de caja?
2. **Estadística y Dispersión:** ¿Cuál es la variabilidad real y la presencia de transacciones atípicas (outliers)?
3. **Comportamiento Temporal:** ¿Existen patrones de estacionalidad por día de la semana, quincena o fin de mes?
4. **Diagnóstico 50/30/20:** ¿Qué tan alineados están los egresos respecto al estándar de salud financiera?

In [ ]:
# ==============================================================================
# 1. IMPORTACIÓN DE LIBRERÍAS Y CONFIGURACIÓN
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Estilo visual moderno para análisis
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

print("Librerías importadas correctamente.")

In [ ]:
# ==============================================================================
# 2. CARGA Y PREPROCESAMIENTO DE DATOS
# ==============================================================================
import os
data_path = os.path.join("..", "data", "demo_finanzas.csv")
df = pd.read_csv(data_path)

# Conversión de tipos
df['Fecha'] = pd.to_datetime(df['Fecha'])
df['Monto'] = pd.to_numeric(df['Monto'], errors='coerce')

# Features temporales derivadas
df['Año'] = df['Fecha'].dt.year
df['Mes'] = df['Fecha'].dt.to_period('M')
df['Dia_Mes'] = df['Fecha'].dt.day
dias_map = {0: 'Lunes', 1: 'Martes', 2: 'Miércoles', 3: 'Jueves', 4: 'Viernes', 5: 'Sábado', 6: 'Domingo'}
df['Dia_Semana'] = df['Fecha'].dt.dayofweek.map(dias_map)
df['Es_Fin_Semana'] = df['Fecha'].dt.dayofweek.isin([5, 6])

print(f"Dimensiones del dataset: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head()

## 3. Resumen Estadístico y Calidad de Datos

In [ ]:
print("=== ESTADÍSTICAS DESCRIPTIVAS DE MONTOS (S/) ===")
stats = df['Monto'].describe().to_frame().T
stats['IQR'] = df['Monto'].quantile(0.75) - df['Monto'].quantile(0.25)
stats['Skewness'] = df['Monto'].skew()
display(stats)

print(f"Valores nulos por columna:\n{df.isnull().sum()}")

## 4. Análisis de Concentración de Gasto (Principio de Pareto 80/20)

In [ ]:
# Agrupación por categoría
pareto_cat = df.groupby('Categoria')['Monto'].sum().sort_values(ascending=False).reset_index()
pareto_cat['Pct_Total'] = (pareto_cat['Monto'] / pareto_cat['Monto'].sum()) * 100
pareto_cat['Pct_Acumulado'] = pareto_cat['Pct_Total'].cumsum()

# Visualización Gráfica de Pareto
fig, ax1 = plt.subplots(figsize=(13, 6))
ax2 = ax1.twinx()

sns.barplot(data=pareto_cat, x='Categoria', y='Monto', ax=ax1, color='#2563eb', alpha=0.85)
ax2.plot(pareto_cat['Categoria'], pareto_cat['Pct_Acumulado'], color='#dc2626', marker='o', linewidth=2.5)
ax2.axhline(80, color='gray', linestyle='--', label='Umbral 80% Pareto')

ax1.set_title("Diagrama de Pareto: Distribución de Gasto por Categoría", fontsize=14, fontweight='bold')
ax1.set_ylabel("Monto Total Acumulado (S/)")
ax2.set_ylabel("Porcentaje Acumulado (%)")
ax1.tick_params(axis='x', rotation=45)
ax2.grid(False)
plt.tight_layout()
plt.show()

top_80 = pareto_cat[pareto_cat['Pct_Acumulado'] <= 85]
print("Categorías que explican ~80% del gasto:", list(top_80['Categoria']))

## 5. Detección Estadística de Valores Atípicos (Outliers)

In [ ]:
# Detección por Rango Intercuartílico (IQR)
q25 = df['Monto'].quantile(0.25)
q75 = df['Monto'].quantile(0.75)
iqr = q75 - q25
limite_superior = q75 + 1.5 * iqr

outliers_df = df[df['Monto'] > limite_superior]
print(f"Umbral superior IQR: S/ {limite_superior:.2f}")
print(f"Nº de transacciones atípicas detectadas: {len(outliers_df)} ({len(outliers_df)/len(df)*100:.1f}% del total)")
print(f"Monto total en transacciones atípicas: S/ {outliers_df['Monto'].sum():,.2f}")

# Boxplot por categoría
plt.figure(figsize=(14, 6))
sns.boxplot(data=df, x='Categoria', y='Monto', palette='Set2')
plt.axhline(limite_superior, color='red', linestyle='--', label=f'Corte Outlier General (S/ {limite_superior:.0f})')
plt.title("Dispersión y Detección de Outliers por Categoría", fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

## 6. Estacionalidad y Patrones Temporales

In [ ]:
orden_dias = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
gasto_dia_semana = df.groupby('Dia_Semana')['Monto'].agg(['mean', 'sum']).reindex(orden_dias)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Gasto promedio por día
sns.barplot(x=gasto_dia_semana.index, y=gasto_dia_semana['mean'], ax=ax1, palette='Blues_d')
ax1.set_title("Ticket Promedio por Día de la Semana", fontweight='bold')
ax1.set_ylabel("Gasto Promedio (S/)")
ax1.tick_params(axis='x', rotation=30)

# Gasto por día del mes (Quincena y fin de mes)
gasto_dia_mes = df.groupby('Dia_Mes')['Monto'].sum()
ax2.plot(gasto_dia_mes.index, gasto_dia_mes.values, marker='o', color='#059669', linewidth=2)
ax2.axvline(1, color='orange', linestyle=':', label='Inicio de mes (Gastos fijos/Alquiler)')
ax2.axvline(15, color='purple', linestyle=':', label='Quincena')
ax2.set_title("Evolución Acumulada por Día del Mes", fontweight='bold')
ax2.set_xlabel("Día del Mes (1 al 31)")
ax2.set_ylabel("Monto Total (S/)")
ax2.legend()

plt.tight_layout()
plt.show()

## 7. Diagnóstico de Salud Financiera: Regla 50 / 30 / 20

In [ ]:
mapa_50_30_20 = {
    'Hogar': 'Necesidades',
    'Comida': 'Necesidades',
    'Transporte': 'Necesidades',
    'Salud': 'Necesidades',
    'Deuda': 'Necesidades',
    'Ocio': 'Deseos',
    'Ropa y Calzado': 'Deseos',
    'Tecnología': 'Deseos',
    'Regalos': 'Deseos',
    'Ahorro/Inversión': 'Ahorro / Futuro',
    'Educación': 'Ahorro / Futuro'
}

df['Pilar_50_30_20'] = df['Categoria'].map(mapa_50_30_20).fillna('Deseos')
resumen_pilares = df.groupby('Pilar_50_30_20')['Monto'].sum()
total_gastos = resumen_pilares.sum()

df_pilares = pd.DataFrame({
    'Monto_Real': resumen_pilares,
    'Porcentaje_Real': (resumen_pilares / total_gastos) * 100,
    'Meta_Teórica': [20.0 if idx == 'Ahorro / Futuro' else (30.0 if idx == 'Deseos' else 50.0) for idx in resumen_pilares.index]
})
df_pilares['Desviacion_vs_Meta'] = df_pilares['Porcentaje_Real'] - df_pilares['Meta_Teórica']
display(df_pilares.round(2))

# Visualización de Brecha vs Meta
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(df_pilares))
width = 0.35

ax.bar(x - width/2, df_pilares['Porcentaje_Real'], width, label='Distribución Real (%)', color='#2563eb')
ax.bar(x + width/2, df_pilares['Meta_Teórica'], width, label='Meta 50/30/20 (%)', color='#10b981')

ax.set_ylabel('Porcentaje (%)')
ax.set_title('Comparativa: Distribución Real de Gastos vs. Estándar 50/30/20', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(df_pilares.index)
ax.legend()
plt.tight_layout()
plt.show()

## 8. Conclusiones y Recomendaciones de Negocio

1. **Concentración de Gasto:** `Comida`, `Transporte` y `Hogar` representan más del 75% del volumen financiero total. Pequeñas optimizaciones porcentuales en el ticket de supermercado o combustible generan ahorros de mayor magnitud que recortar suscripciones menores.
2. **Picos de Fin de Semana:** Se observa un incremento del ticket promedio los días viernes y sábados atribuible a salidas y delivery. Establecer un tope presupuestario semanal para ocio previene desvíos de fin de mes.
3. **Diagnóstico 50/30/20:** El porcentaje de ahorro/futuro actual debe monitorearse para garantizar la meta mínima del 20%, priorizando la constitución del fondo de emergencia de 3 a 6 meses de gastos fijos.